# 第 11 章の実習　日時と時間の変化

2012 年 6 月 27 日，7 月 4 日，7 月 11 日は，ワシントン D.C. の暦でいずれも水曜日です。Capital Bikeshare の全域の記録を使って，どの日にどの時間帯の貸出が多かったか調べます。

Colab では「ドライブにコピーを保存」で自分用のノートを作り，上から準備セルを実行します。CSV は公開 URL から読み込むため，アップロードは不要です。ローカルでは，実習 ZIP を展開してノートブックと CSV を同じフォルダーに置くと，手元のファイルを使えます。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

file_name = "bike-sharing-hourly.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

hourly = pd.read_csv(csv_path, parse_dates=["dteday"])
wednesdays = pd.to_datetime(["2012-06-27", "2012-07-04", "2012-07-11"])
three_days = hourly.loc[hourly["dteday"].isin(wednesdays)]
three_days[["dteday", "hr", "cnt", "holiday"]].head()

## 1. 一日ごとの合計を見る（約 5 分）

時間別の表から日別の件数を作ります。`size()` で時刻行の数，`sum()` で貸出件数を調べます。`holiday` は D.C. の祝日の印です。

日別の結果を確認したら，**8 時台に最も貸出が多い日**を一つ予想し，理由も一文で書きます。

In [ ]:
daily_summary = three_days.groupby("dteday").agg(
    hours_in_table=("hr", "size"),
    rentals=("cnt", "sum"),
    holiday=("holiday", "first"),
)
daily_summary

（ここに予想と理由を書く）

## 2. 時刻別に比べる（約 7 分）

一日の中でいつ貸出が多いかを，同じ横軸と縦軸で比べます。横軸の `8` は 8 時台に始まった貸出です。三つの曲線で，朝が最も高い日と，20 時台以降が高い日を見分けてください。

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.5), layout="constrained")
for date, color, style in zip(
    wednesdays, ["#195b8a", "#a7421f", "#33735d"], ["-", "--", ":"]
):
    hours_for_date = three_days.loc[three_days["dteday"] == date].sort_values("hr")
    ax.plot(hours_for_date["hr"], hours_for_date["cnt"], color=color, linestyle=style,
            marker=".", label=date.strftime("%m-%d"))
ax.set(xlabel="Clock hour", ylabel="Rentals per hour", ylim=(0, 950),
       xlim=(-0.3, 23.3), xticks=[0, 4, 8, 12, 16, 20, 23])
ax.legend(frameon=False, loc="upper left")
plt.show()

three_days.loc[three_days["hr"].isin([8, 17, 21]), ["dteday", "hr", "cnt"]]

## 3. 調べる時間帯を変更する（約 8 分）

8〜10 時台をまとめた合計を出します。次に，セルの最初の二つの値を自分で変更して，20〜22 時台の合計を出してください。`between()` は両端を含むので，どちらの時間帯も 3 時刻分です。

三つの日を同じ順で表示するグラフです。**朝と夜でどの日が一番多かったか**，それぞれの合計とともに記録します。8 時台だけの比較と，8〜10 時台をまとめた比較で，最も多い日が同じかも確認します。最初の予想に戻り，日別・時間帯別の結果の両方を使って，朝の対応を考える人への説明を 2〜3 文で書きます。

In [ ]:
first_hour = 8
last_hour = 10

in_window = three_days["hr"].between(first_hour, last_hour)
window_totals = three_days.loc[in_window].groupby("dteday")["cnt"].sum().reindex(wednesdays)
window_totals.name = "rentals"

fig, ax = plt.subplots(figsize=(5.8, 3.0), layout="constrained")
ax.bar(window_totals.index.strftime("%m-%d"), window_totals,
       color=["#195b8a", "#a7421f", "#33735d"])
ax.set(xlabel="Date (2012)", ylabel="Rentals in selected hours",
       title=f"Clock hours {first_hour:02d}–{last_hour:02d}",
       ylim=(0, max(1800, 1.08 * window_totals.max())))
plt.show()

window_totals

（ここに朝と夜の比較，説明を書く）

### 理由を調べる（任意）

7 月 4 日は独立記念日です。国立公園局に，当日の夜の花火の記録があります。貸出の内訳を `cnt` から `registered` か `casual` へ変更し，同じ時間帯の比較で日付の順位がどう変わるか確かめます。夜の利用の理由をさらに調べるために必要な資料を一つ挙げてください。 [D.C. 人事局の 2012 年祝日表](https://dchr.dc.gov/sites/default/files/dc/sites/dchr/publication/attachments/DCHR_edpm_12_261_schedules_holidays_2012_2013.pdf) ・ [米国国立公園局の花火の記録](https://www.nps.gov/media/photo/gallery-item.htm?gid=AD5299B3-1DD8-B71B-0BADBF711BBDD03B&id=ad800bf9-1dd8-b71b-0b28-ff00da238951)

提出する場合はカーネルを再起動し，上から全セルを実行して，選んだ時間帯の図と説明が残る状態で保存します。

### 出典

H. Fanaee-T, *Bike Sharing*, UCI Machine Learning Repository（CC BY 4.0）。Capital Bikeshare の 2011〜2012 年の実測ログを研究者が集計した時間別の表を使いました。CSV は原配布の `hour.csv` の名前と改行コードを変更したものです。 [データ DOI](https://doi.org/10.24432/C5W894) ・ [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)